# Exponential propagation with a periodically driven input
Integrating homogeneous decay exactly still leaves a numerical choice for the input. Compare a frozen heater signal with its exact convolution. Time is in seconds and the normalised temperature is in kelvin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

decay=.8                         # per second
frequency=1.                     # cycles per second
samples_per_period=1
cycles=8
assert .1 <= decay <= 4 and .25 <= frequency <= 4 and 1 <= samples_per_period <= 64 and 2 <= cycles <= 24
w=2*np.pi*frequency
h=1/(samples_per_period*frequency)
drive=1.                         # kelvin per second
t=np.arange(cycles*samples_per_period+1)*h
def exact(t):
    return drive*np.real((np.exp(1j*w*t)-np.exp(-decay*t))/(decay+1j*w))
def freeze(t,dt):
    out=[0.]
    gain=-np.expm1(-decay*dt)/decay
    for start in t[:-1]:
        out.append(np.exp(-decay*dt)*out[-1]+gain*drive*np.cos(w*start))
    return np.array(out)
approx=freeze(t,h)
continuous=exact(t)
print(f"Final temperature: frozen {approx[-1]:.7f}; exact {continuous[-1]:.7f} K")
if samples_per_period==1:
    assert np.allclose(approx,drive*(1-np.exp(-decay*t))/decay)

## Resolve the source in time
The homogeneous exponential factor is stable for every step shown. Use a finer time grid to see which periodic input the coarse samples missed. The convergence panel holds the final physical time fixed.

In [ ]:
fig,(values,refine)=plt.subplots(1,2,figsize=(10,4),layout="constrained")
tt=np.linspace(0,t[-1],801)
values.plot(tt,exact(tt),color="#145c71",label="exact convolution")
values.plot(t,approx,"o--",ms=3,color="#b94626",label="frozen forcing")
values.set(xlabel="Time (s)",ylabel="Temperature change (K)",title="Exact decay, sampled heat input")
values.legend(fontsize=8)
nn=2**np.arange(0,8)
err=[]
for n in nn:
    dt=1/(frequency*n)
    sample=np.arange(cycles*n+1)*dt
    err.append(np.max(abs(freeze(sample,dt)-exact(sample))))
refine.loglog(nn,err,"o-",color="#145c71")
refine.set(xlabel="Samples per forcing period",ylabel="Max sampled error (K)",title="Refinement restores the waveform")
for ax in (values,refine):ax.grid(alpha=.2)
plt.show()